In [0]:
from pyspark.sql.functions import *

# 1. Read from Silver table
silver_df = spark.table("workspace.ibm.capstone_silver_sales1")

# 2. Aggregate data for Gold Layer (supporting monthly, state, and category insights)
gold_df = (
    silver_df
    .groupBy("year", "month", "month_name", "state", "category")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("units_sold"),
        sum("gross_amount").alias("gross_sales"),
        sum("discount_amount").alias("discount_amount"),
        sum("net_amount").alias("net_sales"),
        avg("net_amount").alias("average_order_value")
    )
    .orderBy("year", "month", "state", "category")
)

# 3. Write to Gold Delta Table (with overwriteSchema to prevent mismatch errors)
gold_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.ibm.capstone_gold_sales_summary1")

# Preview Gold analytical data
display(gold_df)